In [46]:
### Import packages
import sys, getopt, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler #StandardScaler is sensitive to outlier

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.algorithms import QSVR
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [ ]:
#py
#System utilities
#sys, getopt, os are python libraries

# Think of sys as Python's communication channel with the operating system.
#It lets Python know things like Python version , command-line argumentssystem paths, memory-related information 
# eg; import sys
#print(sys.version) , Output: Python 3.11.4 
# Remark: Not really useful here

#getopt
#This is for reading command-line arguments.
#Example:python train.py --epochs 50
#getopt helps interpret options like --epochs.
# Remark: Not useful

#os
#os stands for Operating System.
#It allows Python to interact with files and folders.
#Examples:
#os.getcwd()
#Current working directory.
#os.listdir()
#List files.
#os.mkdir()
#Create a folder.
#Remark : ML deals with datasets, so 'os' is usually imported

#numerical computing
#import numpy as np
#imports numpy as np

#Data handling
#import pandas as pd
#imports pandas as pd

#Visualization
#import matplotlib.pyplot as plt
#imports matplotlib as plt

#CML
#from sklearn.model_selection import RepeatedKFold
#scikitlearn is a classical ml library used in python
#here due to the reduced number of sets on our hands, we use "RepeatedKfold".
#This repeats the entire K-fold process multiple times using different random splits.That gives a more reliable estimate of model performance.
#This is especially valuable in small materials-science datasets, where results can vary a lot depending on how the data is split.

#from sklearn.metrics import mean_squared_error, r2_score
# mse and r2 are used to evaluate the error of the calculations

#from sklearn.preprocessing import MinMaxScaler, StandardScaler 
#This prepares the data before training.Machine learning models often perform much better when features are on similar scales.

#MinMaxScaler:Transforms each feature into a fixed range, typically:0 → 1
#Remark: Useful when you want every feature bounded.

#StandardScaler:Transforms each feature to have:mean = 0, standard deviation = 1
#Many algorithms (including SVM-based methods) work well with standardized data.
#The author's comment says: "# StandardScaler is sensitive to outlier"
#This is a reminder that extreme values can strongly influence the mean and standard deviation, making the scaling less representative.

#QML
#from qiskit.circuit.library import ZZFeatureMap
#A feature map answers the question:"How do we take ordinary classical data (numbers from a CSV file) and encode it into a quantum circuit?""
#ZZFeatureMap is a predefined encoding circuit in Qiskit that uses both single-qubit rotations and entangling interactions. 
# It is widely used in quantum kernel methods, including QSVC.

#from qiskit_machine_learning.algorithms import QSVR
#This is the model that the paper is built around.QSVC stands for Quantum Support Vector Regressor.
#It's the quantum analogue of a classical Support Vector Regressor, but instead of using a classical kernel, it relies on a quantum kernel computed from quantum circuits.

#from qiskit_machine_learning.kernels import FidelityQuantumKernel
#This is the component that computes the quantum kernel.A kernel measures how similar two data points are.
#In classical SVMs, kernels are computed with mathematical formulas (linear, polynomial, RBF, etc.).
#Here, similarity is measured using quantum state fidelity—how similar the encoded quantum states are after passing through the feature map.
#This kernel is then passed into QSVR, which uses it to perform classification.

In [59]:
#Instead of FEATURE_MAP_REPS_LIST = [1,2,3,4,5] and REGU_PARA_LIST = [0.1,1,10,100],EPISLON_LIST = [0.01, 0.001], I used FEATURE_MAP_REPS_LIST = [1],REGU_PARA_LIST = [0.1], EPISLON_LIST = [0.01] to reduce the number of experiments/Iterations and save time. You can change it back to the original values if you want to run more experiments.
#Also I changed the number of repeats from N_REPEATS = 10 to N_REPEATS = 1 for the same reason as above. You can change it back to the original value if you want to run more experiments.

#This can be used to test the code and make sure it works, and then you can change it back to the original values to run the full set of experiments.

In [ ]:
root_folder = 'QSVR'
### Globals
# For reproducibility
np.random.seed(42)

# Fixed feature sizes
NUM_FEATURES = 3
NUM_QUBITS = NUM_FEATURES
NUM_TARGETS = 1

# Quantum circuit parameters
FEATURE_MAP_REPS_LIST = [1,2,3,4,5]
REGU_PARA_LIST = [0.1,1,10,100]
EPISLON_LIST = [0.01, 0.001]
ENTANGLEMENT_LIST = ['linear', 'full', 'circular']

# Training hyperparameters
#LEARNING_RATE = 0.01
#BATCH_SIZE = 30
#NUM_EPOCHS = 100 # Adjust as needed

# K-fold cross-validation parameters
N_REPEATS = 10
TEST_SIZE = 1

In [ ]:
#root_folder = 'QSVC'
#creates a folder named QSVC and stores the results, graphs, predicition etc there.
### Globals ; says that the ones below are global variables(This can be used throughout the notebook without redefining it. eg: NUM_FEATURES)

#np.random.seed(42)
#A random seed fixes the random number generator.Think of it like telling Python:
#"Whenever you need random numbers, start from exactly this point."
#So every time the notebook runs:
#       42
#       ↓
# Same random numbers
#       ↓
#Same train/test split
#       ↓
#   Same results
#Remark: Important for Reproducibility


#NUM_FEATURES = 3
#Means, The model will use 3 input features. A feature is an input variable.
#Here, the variables are : Electronegativity, Bulk Modulus, Atomic Volume

#NUM_QUBITS = NUM_FEATURES
#connects classical and quantum ml.
#The feature map (ZZFeatureMap) encodes classical features into quantum states.
# 3 features = 3 qubits : feature1 = qubit1, feature2 = qubit2, feature3 = qubit3

#NUM_TARGETS = 1
#This says that, input = 3 features; output = 1 target(stacking fault energy)

#FEATURE_MAP_REPS_LIST = [1,2,3,4,5]; Feature Map Repetition list
#A feature map converts classical numbers into a quantum state(Translates classical data into quantum circuit.).
#Reps means repeating the same process over and over.
#[1,2,3,4,5] means the authors are not choosing one value.They intend to test five different quantum circuits.
#This is called a hyperparameter sweep.
#The notebook will probably loop over
#reps = 1
#reps = 2
#reps = 3
#reps = 4
#reps = 5
#to see which performs best.

#REGU_PARA_LIST = [0.1,1,10,100]; Regularization Parameters list
#The regularization parameter controls how flexible the Support Vector Machine is.
#Imagine trying to separate two classes.
#A very flexible model can create complicated decision boundaries.A more regularized model prefers simpler boundaries.
#The authors want to test four different settings. 0.1,1,10,100
#the authors ask "Which regularization value gives the best performance?"


#ENTANGLEMENT_LIST = ['linear', 'full', 'circular']
#Entanglement determines which qubits are allowed to interact.Different interaction patterns create different quantum feature spaces.
#Linear: Q0 —— Q1 —— Q2 . Each qubit only talks to its neighbours.
#Full       Q0 —— Q1
#            |\    |
#            | \   |
#            |  \  |
#            Q2 ----
#Every qubit interacts with every other qubit. Most expressive. Most expensive.
#Circular Q0 —— Q1
#          |     |
#          Q2 ----
#Like a ring.The first and last qubits are also connected.
#The structure of entanglement affects the quantum kernel.The authors want to discover which topology gives the best classification performance.

#N_REPEATS = 10 ; The notebook will perform repeated cross-validation ten times.
#Different training samples can produce different accuracies.
#Instead of trusting one split,the authors repeat the entire experiment ten separate times.
#That produces a much more reliable estimate.

#What is new compared to QSVC
#Unlike QSVC, there is not classifier threshold here. 
#In Classification, The model ask Correct? or Incorrect? Only two possibilities.
#In Regression, Predictions are continuous. EPSILON plays a significant role here. Say EPSILON = 0.1, 
#then that value is compared with the error , if its lower than that, its ignored and if its higher, its punished.
# This is called the 'ε-insensitive loss' , It is one of the defining ideas of Support Vector Regression.
# The authors are testing two values, 0.1 and 0.01, because they don't know which will be better.
# As if the value is high, its more forgivable and if the value is small, its more stricter. 


In [48]:
#Dataset preparation

def prepare_dataset_k_fold(X, y, train_indices, test_indices):
    # Separate train/test split
    X_train_raw, X_test_raw = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    # Separate element column from the actual features
    element_test = X_test_raw[:, 0]
    element_train = X_train_raw[:, 0]

    # Drop the element column (first column)
    X_train = X_train_raw[:, 1:]
    X_test = X_test_raw[:, 1:]

    full_X = np.vstack([X_train, X_test])

    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(full_X)

    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, y_train, X_test_scaled, y_test, element_test, element_train

In [ ]:
#It prepares one train-test split for cross-validation by separating the data, removing unnecessary columns, scaling the numerical features, and returning everything needed for training and evaluation.
#Arguments = (X, y, train_indices, test_indices)

#X.This is the input feature matrix.Imagine the dataset before processing.
#Element	Electronegativity	Bulk Modulus	Volume
#Mg-Al	          1.42	             37	         14.2
#Mg-Zn	          1.65	             41	         13.8
#Mg-Y	          1.22	             28	         18.1

#y.This contains the labels.
#For classification:
#0
#1
#0
#1
#These are the answers the model is trying to learn.

#train_indices
#Suppose Repeated K-Fold decides
# Training, Samples
#0
#2
#3
#5
#6
#8

#It doesn't copy those rows.Instead it simply stores [0,2,3,5,6,8].
#Those are the training indices.

#test_indices
#Similarly, [1,4,7] means
#Rows
#1
#4
#7
#are the testing set.

#X_train_raw, X_test_raw = X[train_indices], X[test_indices]
# y_train, y_test = y[train_indices], y[test_indices]
#Train Test split

#   element_test = X_test_raw[:, 0]
#   element_train = X_train_raw[:, 0](Splicing)
#   Only returns the first coloumn of data, which is the element name

#   X_train = X_train_raw[:, 1:]
#    X_test = X_test_raw[:, 1:]
#removes the first coloumn, which has the names of the elements(strings) from the further calculations.

#full_X = np.vstack([X_train, X_test])
#np.vstack means vertical stack.
#Training
#1 2 3
#4 5 6
#Testing
#7 8 9

#After
#np.vstack(...)
#you obtain
#1 2 3
#4 5 6
#7 8 9
#The two datasets are temporarily joined together.

# scaler = MinMaxScaler(feature_range=(-1, 1))
# This creates a MinMaxScaler object.It doesn't scale anything yet.
# Think of it like buying a ruler.The ruler exists, but you haven't measured anything.
#                   Feature	            Range 
#               Electronegativity	    1–4
#                Bulk Modulus	        20–250
#                    Volume	            10–30

#Notice that one feature has values around 200 while another is around 2.
#If we leave them like this, larger numerical ranges can dominate the calculations.
#Scaling brings all features onto a common scale.

# scaler.fit(full_X)
# This line computes the minimum and maximum values of each feature.The scaler stores these values internally.
# fit() does not modify the data. It only learns the transformation.

#X_train_scaled = scaler.transform(X_train)
#X_test_scaled = scaler.transform(X_test)
#the learned scaling is applied to the training features.The same transformation is then applied to the test features.
#Using the same scaler is essential so that training and testing data remain in the same feature space.

#The choice to fit the scaler on full_X (training + testing) is perfectly acceptable to reproduce the published work exactly
#But if you later develop your own improved methodology, one potential enhancement would be to fit the scaler using only the training data within each cross-validation fold. 
#Recognizing these small design decisions is an important step from simply reproducing code to critically evaluating and eventually improving it.

In [49]:
#Quantum kernel builder

def reconfig_quantum_kernel_qsvr(feature_dimension, epsilon, C, reps, entangle):
    """
        Create a quantum kernel
        qsvr = QSVR(C=20.0, epsilon=0.2, quantum_kernel=kernel)

        Args:
            feature_dimension: Dimension of the feature space.
            reps: Number of repetitions of quantum circuit.
            C: Regularization parameter.
               The strength of the regularization is inversely proportional to C.
               Must be strictly positive. The penalty is a squared l2.
            epsilon: Epsilon in the epsilon-SVR model.
                     It specifies the epsilon-tube within which no penalty is associated in the training
                     loss function with points predicted within a distance epsilon from the actual value.
                     Must be non-negative.
            entangle: Entanglement type of the feature map.

        Returns:
            qsvr: quantum kernel
    """
    feature_map = ZZFeatureMap(feature_dimension=feature_dimension, reps=reps, entanglement=entangle, insert_barriers=True)
    kernel = FidelityQuantumKernel(feature_map=feature_map)
    qsvr = QSVR(C=C, epsilon=epsilon, quantum_kernel=kernel)
    return qsvr

In [ ]:
'''epsilon: Epsilon in the epsilon-SVR model.
                     It specifies the epsilon-tube within which no penalty is associated in the training
                     loss function with points predicted within a distance epsilon from the actual value.
                     Must be non-negative.'''

#Given some hyperparameters, build a Quantum Support Vector Classifier (QSVC) ready for training.
#Takes 3 features and gives 1 QSVC model.
#(feature_dimension, C, reps, entangle)

#earlier we saw NUM_FEATURE = 3, so feature_dimension should be = 3
#The map creates 3 encoded inputs for 3 features.

#C comes from classical SVM
#A small C value means it is more tolerant of mistakes and a large C value means its less tolerant of mistakes.
#The comment says : Regularization is inversely proportional to C

#Regularization tries to prevent overfitting.
#Higher regularization --> simpler model --> less chance of memorizing noise.
#This is why the notebook tests 0.1,1,10,100. The authors don't know beforehand which value works best.

#reps
#Mentioned earlier as how many times the feature map repeats.

#entangle
#This controls how qubits talk to each other.
#Possible values linear, full, circular.

# feature_map = ZZFeatureMap(feature_dimension=feature_dimension, reps=reps, entanglement=entangle, insert_barriers=True)
# Feature maps perform a translation from classical data to quantum circuit.
#The paper chose ZZFeatureMap, because it introduces entanglement between qubits, allowing the encoded state to capture relationships between features rather than treating each feature independently.
#insert_barriers=True: A barrier is simply a visual separator in a quantum circuit diagram.

# kernel = FidelityQuantumKernel(feature_map=feature_map); This line creates the quantum kernel.
# Suppose we have two alloys.Mg-Al, Mg-Zn.A kernel answers: "How similar are these two?".
# Encodes both alloys into quantum states.
# Then computes their fidelity, means how similar the alloys are(High fidelity means they are very similar and vice versa).
# So the quantum kernel replaces the classical similarity function with one derived from quantum circuits.

#qsvc = QSVC(C=C, quantum_kernel=kernel)
#Now we plug that kernel into QSVC.
#The classifier itself is still based on the familiar SVM framework.
#The "quantum" part comes from how similarity between samples is computed, not from replacing the entire algorithm.
#QSVC uses that quantum-derived similarity matrix to build a classical support vector classifier.

# return qsvc
# Finally, the function returns the fully constructed model.
# Notice that it does not train the model. It only builds it.

In [50]:
#Training function 

def train_qsvr(qsvr, X_train, y_train, X_test):
    """
        Train based on X_train/y_train (after scaling), return prediction from X_test

        Args:
            qsvr: quantum kernel
            X_train:
            y_train
            X_test

        Returns:
    """
    qsvr.fit(X_train, np.concatenate(y_train))
    return qsvr.predict(X_train), qsvr.predict(X_test)

In [ ]:
#This function is the point where the model goes from an empty model to a trained classifier.

#def train_qsvc(qsvc, X_train, y_train, X_test):
#Inputs: QSVC Model,Training Data, Training Labels, Testing Data
#Output: Predictions on Training Set, Predictions on Testing Set
#The function doesn't return the trained model itself.Instead it returns the predictions.

#qsvc; This is the model we created earlier.
#At this point, it already knows;
#                     *which feature map to use
#                     *which quantum kernel to use
#                     *which value of C to use
#But it doesn't know anything about the dataset yet.

#X_train
#These are the training features.
#Example

'''   EN	Bulk Modulus	Volume
    -0.3	    0.65	   -0.10
    0.25	    -0.18	    0.41      '''
# The values have been scaled to -1,1

#y_train
#These are correct answers for the model to train: 10101

#X_test
#These are the unseen samples.The model has never looked at them.They are used only for evaluation.

#qsvc.fit(X_train, np.concatenate(y_train))
#  .fit() , means learning from data.
#which means now, it learned how to classify data.
#the quantum computer(or simulator)is not classifying directly.
#Instead,it computes the kernel matrix.The SVM then solves its optimization problem.
#np.concatenate(y_train) : changes the shape
#concatenate changes the array that is in coloumn vector form to row form , which the QSVC expects.
#Once this line has run, the model has learned. Now, its time to predict.

#return qsvc.predict(X_train), qsvc.predict(X_test)
# .predict() uses the learned data and predicts
# the function returns two arrays, one contains training predictions and other testing predictions.
#This function only answers "What classes did the model predict?"
#This tiny function hides almost all of the computational work of the QSVC algorithm. 
#When you call fit(), you're not simply fitting a classical SVM. 
#Qiskit first uses the ZZFeatureMap to encode every training sample into a quantum state, computes the quantum kernel matrix using state fidelities, and then hands that kernel matrix to a classical Support Vector Machine optimizer. 
#Once that optimization is complete, the model can classify both the training and unseen test samples through predict().


In [51]:
#command line argument parser

def get_arguments(argvs):
    _entangle = ''
    _feature_map_reps = ''
    _regu_para = ''
    _epsilon = ''
    try:
        opts, args = getopt.getopt(argvs, "h:e:f:r:p:", ["entangle=", "feature_map_reps=", "_regu_para=", "_epsilon="])
    except getopt.GetoptError:
        print(root_folder + '.py -e <entangle> -f <feature_map_reps> -r <regu_para> -p <epsilon>')
        sys.exit(2)
    for opt, arg in opts:
        if opt == '-h':
            print(root_folder + '.py -e <entangle> -f <feature_map_reps> -r <regu_para> -p <epsilon>')
            sys.exit()
        elif opt in ("-e", "--entangle"):
            _entangle = arg
        elif opt in ("-f", "--feature_map_reps"):
            _feature_map_reps = int(arg)
        elif opt in ("-r", "--regu_para"):
            _regu_para = float(arg)
        elif opt in ("-p", "--epsilon"):
            _epsilon = float(arg)
    return _entangle, _feature_map_reps, _regu_para, _epsilon


In [ ]:
#This function is software engineering. 
#It doesn't affect the QSVR algorithm itself; it simply makes the code easier to run from the command line.

#def get_arguments(argvs):
#Purpose: Read options provided by the user when running the program from the terminal.

#Suppose instead of opening a Jupyter notebook, you have a Python script called QSVR.py
#You could run it like this:
#python QSVR.py -e linear -f 3 -r 10
'''This command means:

Run QSVR.py

Use:
    entanglement = linear
    feature map repetitions = 3
    regularization = 10 '''

# Without this function, Python would have no idea what -e, -f, -r mean.

#argvs : argument vector
#It is simply a list of everything typed after the program name.(LIke decoding the above example)

#Creates variable
'''_entangle = ''
    _feature_map_reps = ''
    _regu_para = '' 
     __epsilon = '' 
                         '''
# these are containers waiting to be filled.

#try and except are used so that the code isnt crashed. 

# opts, args = getopt.getopt(argvs, "h:e:f:r:p:", ["entangle=", "feature_map_reps=", "_regu_para="])
# getopt was in the import section. It looks through args and seperates opts and args

#h:e:f:r: define the short command line flags. h- help, e- entanglement, f- Feature map repetitions, r- Regularization parameter


#From a Quantum Machine Learning perspective, this function doesn't change the algorithm at all. 
#Whether you build a QSVC through hard-coded values or command-line arguments, the mathematics is identical.

#From a research software perspective, however, this function is very useful.
#Imagine the authors wanted to launch dozens of experiments on a Linux workstation or HPC cluster. Instead of editing the notebook every time, they could run commands such as:
'''
python QSVC.py -e linear -f 1 -r 0.1
python QSVC.py -e linear -f 2 -r 0.1
python QSVC.py -e full -f 5 -r 100
                                        '''
#This allows experiments to be automated with shell scripts or job schedulers. 
#In fact, this function hints that the notebook was probably adapted from a standalone Python script used to run many QSVR experiments systematically.

In [ ]:
#output folder
date = "24_19_25_1"

if not os.path.exists(f'{root_folder}/result'):
    os.makedirs(f'{root_folder}/result')
if not os.path.exists(f'{root_folder}/logs'):
    os.makedirs(f'{root_folder}/logs')

In [ ]:
#This is another software engineering / research organization cell. 
#It has nothing to do with the QSVC algorithm itself, but it's an excellent example of how researchers keep their experiments organized.
#The previous cell deals with the input and this cell deals with the output.
""" if not os.path.exists(f'{root_folder}/result'):
    os.makedirs(f'{root_folder}/result')
if not os.path.exists(f'{root_folder}/logs'):
    os.makedirs(f'{root_folder}/logs') """

''' Start Program
      │
      ▼
Is "QSVR/result" present?
      │
 ┌────┴────┐
 │         │
Yes       No
 │         │
Skip    Create Folder
 │
 ▼
Is "QSVR/logs" present?
      │
 ┌────┴────┐
 │         │
Yes       No
 │         │
Skip    Create Folder
 │
 ▼
Continue Experiment '''

#Although this cell contains no machine learning or quantum computing, it reflects good computational research practice.
#Scientific experiments often generate dozens or even hundreds of output files—accuracy tables, prediction CSVs, plots, and log files. 
#By automatically creating the required directory structure, the authors make the code portable and reproducible across different computers.

In [60]:
#Load dataset

dataset_name = "/home/ashok/Desktop/ABHI/Learning,Reproducing/qml_training-validation-data.csv"
df = pd.read_csv(dataset_name)
display(df.head())
X = df[['Element', 'el_neg', 'B/GPa', 'Volume/A^3']].values
y = df['SFE/mJm^-3'].values
print(df.shape)

,Element,el_neg,B/GPa,Volume/A^3,SFE/mJm^-3
0,Be,1.57,130.0,8.09,23.48
1,Sc,1.36,57.0,25.00,16.16
2,Ti,1.54,110.0,17.60,24.44
3,Co,1.88,180.0,11.00,37.64
4,Zn,1.65,70.0,15.20,20.98


(21, 5)


In [ ]:
#dataset_name = "./Desktop/ABHI/Learning,Reproducing/qml_training-validation-data.csv"
#designates the location of the dataset to a variable called dataset name.
#using .read_csv, the program reads the dataset and stores in df.
#display over print is used to form table instead of text format.

#X is input features and y is target variable.
#'Element', 'el_neg', 'B/GPa', 'Volume/A^3' These are selected for input features
#double square brackets means select multiple coloumns.
#SFE/mJm^-3 is the target variable.
#single bracket means only one coloumn is needed

# .values convert pandas object to numpy object.
# The model takes 'Element', 'el_neg', 'B/GPa', 'Volume/A^3' and predicts SFE/mJm^-3.
# Electronegativity, Bulk Modulus, Volume ---> Stacking Fault Energy

#.shape is a funtion in pandas to see the size of the dataset
# here its said (21,5) means it has 21 rows and 5 coloumns.

In [54]:
#Scaling the target variable y to the range [-1, 1] using MinMaxScaler
y_scaler = MinMaxScaler(feature_range=(-1, 1))
y = y_scaler.fit_transform(y.reshape(-1, 1))

In [ ]:
#reshape(-1, 1) converts 
#    0
#    1
#    1
#    0
#into [
#      [0]
#      [1]
#      [1]
#      [0]
#          ]

# After .fit_transform, the range becomes 0,1 to -1,1.

#Difference: No Classifier Threshold.

In [55]:
#Cross-validation through RepeatedKFold
rkf = RepeatedKFold(n_splits=X.shape[0] // TEST_SIZE, n_repeats=N_REPEATS)
print(rkf)

RepeatedKFold(n_repeats=1, n_splits=21, random_state=None)


In [ ]:
#rkf = RepeatedKFold(n_splits=X.shape[0] // TEST_SIZE, n_repeats=N_REPEATS)
#This line creates the cross-validation object.
#It does NOT train the model.It simply creates a plan that says,
# "Here's how we'll split the data."

#Cross-validation: Imagine you have 100 alloys.One approach is 80 Training,20 Testing
#Train once.Test once.Done.
#This could create randomness since the dataset is small
#Instead, machine learning repeats the experiment using different train/test splits.This is called Cross Validation.

#RepeatedKFold: .It splits your dataset into k equal parts (folds),
#trains the model on k-1 folds, tests on the remaining fold, 
#and repeats this process multiple times using a randomized data split for each cycle.

#n_splits: how many folds should i divide the coloumn into, since X.shape contains 21 materials,
#it will split into 21 folds. TEST_SIZE was 1 previously.
#Means each split contains 1 test sample.
# This is called Leave-One-Out Cross Validation (LOOCV).
#This is one of the rigourous evaluation startegy for small datasets.

#n_repeats=N_REPEATS means we repeat it using the value we designated earlier, which is 10/1.
#print(rfk): RepeatedKFold(n_repeats=1, n_splits=21, random_state=None)



In [56]:
df = pd.DataFrame(columns=['C', 'reps', 'epsilon', 'entanglement',
                            'element test', 'actual test', 'predicted test',
                            'element train', 'actual train', 'predicted train',
                            'R2 test', 'R2 train'])

In [ ]:
#Creates a table for the data to be stored.
""" Train QSVC
      │
      ▼
Predict Test Sample
      │
      ▼
Store

C

Reps

Epsilon

Entanglement

Test Alloy

True Label

Prediction
↓
Repeat
↓
Large Results Table
↓
Statistical Analysis
↓
Paper Figures"""

''' This DataFrame is essentially the experiment logbook for the entire study. 
Rather than storing only the final average accuracy,
the authors preserve the configuration (C, reps, entanglement),
the identity of the alloy being tested, the true and predicted labels, and performance metrics. 
This makes the experiments traceable and reproducible,
allowing later analysis such as identifying difficult alloys or determining which hyperparameter combinations perform best. '''




In [57]:
# Build output filename

if len(FEATURE_MAP_REPS_LIST) == 1:
    FEATURE_MAP_REPS_LIST_NAME = FEATURE_MAP_REPS_LIST[0]
else:
    FEATURE_MAP_REPS_LIST_NAME = FEATURE_MAP_REPS_LIST

if len(REGU_PARA_LIST) == 1:
    REGU_PARA_LIST_NAME = REGU_PARA_LIST[0]
else:
    REGU_PARA_LIST_NAME = REGU_PARA_LIST

if len(ENTANGLEMENT_LIST) == 1:
    ENTANGLEMENT_LIST_NAME = ENTANGLEMENT_LIST[0]
else:
    ENTANGLEMENT_LIST_NAME = ENTANGLEMENT_LIST

if len(EPISLON_LIST) == 1:
    EPISLON_LIST_NAME = EPISLON_LIST[0]
else:
    EPISLON_LIST_NAME = EPISLON_LIST

file_name = (
    f"{root_folder}/result/"
    f"FMR_{FEATURE_MAP_REPS_LIST_NAME}_"
    f"R_{REGU_PARA_LIST_NAME}_"
    f"E_{ENTANGLEMENT_LIST_NAME}_"
    f"EP_{EPISLON_LIST_NAME}_{date}.csv"
)

print(file_name)

QSVR/result/FMR_1_R_0.1_E_['linear', 'full', 'circular']_EP_0.01_24_19_25_1.csv


In [ ]:
#this cell is not about Quantum Machine Learning. Instead, it's about experiment management.
#Aim of this cell: Generate a descriptive filename that tells you exactly which experiment produced the results.

#The whole block of code leads to how to generate the names for the experiment.
#While reading the file name you can also understand/know about the parameters embedded in that.
#eg:QSVC/result/FMR_1_R_0.1_E_['linear', 'full', 'circular']_EP_0.01_8_19_25_1.csv
#while reading this we know that Feature Map Reps is 1, Regu Para list is 0.1, entanglement is 'linear', 'full', 'circular', Epsilon is 0.01.

In [58]:
i = 0

print("\n--- Start K-Fold Loop ---")

for train_indices, test_indices in rkf.split(X):
    X_train, y_train, X_test, y_test, element_test, element_train = prepare_dataset_k_fold(X, y, train_indices, test_indices)
    for C_value in REGU_PARA_LIST:
        for feature_map_reps in FEATURE_MAP_REPS_LIST:
            for epsilon_value in EPISLON_LIST:
                for entanglement in ENTANGLEMENT_LIST:
                    print(f'REGU_PARA:{C_value} feature_map_reps:{feature_map_reps} '
                            f'epsilon:{epsilon_value} entanglement:{entanglement}')
                    # conf kernel
                    qsvr = reconfig_quantum_kernel_qsvr(feature_dimension=NUM_FEATURES,
                                                        epsilon=epsilon_value,
                                                        C=C_value,
                                                        reps=feature_map_reps,
                                                        entangle=entanglement)

                    # train
                    predict_train, predict_test = train_qsvr(qsvr, X_train, y_train, X_test)

                    # some conversions
                    all_preds = np.array(predict_test)
                    all_targets = np.array(y_test)
                    all_preds = y_scaler.inverse_transform(all_preds.reshape(-1, 1))
                    all_targets = y_scaler.inverse_transform(all_targets.reshape(-1, 1))

                    all_preds_train = np.array(predict_train)
                    all_targets_train = np.array(y_train)
                    all_preds_train = y_scaler.inverse_transform(all_preds_train.reshape(-1, 1))
                    all_targets_train = y_scaler.inverse_transform(all_targets_train.reshape(-1, 1))

                    # save data
                    new_row = {'C': C_value,
                                'reps': feature_map_reps,
                                'epsilon': epsilon_value,
                                'entanglement': entanglement,
                                'element test': element_test,
                                'actual test': np.array(all_targets).flatten(),
                                'predicted test': np.array(all_preds).flatten(),
                                'element train': element_train,
                                'actual train': np.array(all_targets_train).flatten(),
                                'predicted train': np.array(all_preds_train).flatten(),
                                #'R2 test': r2_score(y_test, predict_test),
                                'R2 train': r2_score(y_train, predict_train),
                                }
                    df.loc[len(df)] = new_row
                    with np.printoptions(linewidth=10000):
                        df.to_csv(file_name, index=False)  # update csv every loop
                    df.at[0, "info"] = [f"DATASET: {dataset_name}"]
                    i += 1


--- Start K-Fold Loop ---
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:linear
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:full
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:circular
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:linear
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:full
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:circular
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:linear
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:full
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:circular
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:linear
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:full
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:circular
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:linear
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entanglement:full
REGU_PARA:0.1 feature_map_reps:1 epsilon:0.01 entan

In [ ]:
# The heart of the code
""" It contains:

* Cross Validation
* Hyperparameter Search
* Quantum Model Construction
* Model Training
* Prediction
* Result Collection
* Saving Results 
                    """

#What is the code trying to answer?
#Which quantum regression model predicts stacking fault energy most accurately?
#Instead of trying one model,it tries
'''
Different C values
Different feature map depths
Different epsilon values
Different entanglement strategies '''
#Then compares all of them.

#Big picture
#Entire Dataset --> Cross Validation --> Training-Testing split --> Try every C --> Try every Feature Map --> Try every ε
#Try every Entanglement -->Build QSVR --> Train QSVr --> Predict SFE  --> Store Results -->
#Repeat for next split

#This is almost exactly the same workflow as QSVC. Only one extra loop appears.


In [ ]:
#The First Loop
#for train_indices, test_indices in rkf.split(X):
#rkf: Every iteration gives Training Indices, Testing Indices.All samples get tested here.
# X_train, y_train, X_test, y_test, element_test, element_train = prepare_dataset_k_fold(X, y, train_indices, test_indices)
#It performs Split, Remove element names, Scale features and returns
''' Training Features

    Training Labels

    Testing Features

    Testing Labels

    Training Elements

    Testing Elements '''

In [ ]:
#The Second Loop, the loop over C
#for C_value in REGU_PARA_LIST:
#On the second cell, we've seen that REGU_PARA_LIST:[0.1,1,10,100]
#Therefore, the loop becomes four experiments
''' C = 0.1

        ↓

    C = 1

        ↓

    C = 10

        ↓

    C = 100 
                '''

In [ ]:
#The third loop, loop over repetitions
#for feature_map_reps in FEATURE_MAP_REPS_LIST:
#On the second cell, we've seen that FEATURE_MAP_REPS_LIST:[1,2,3,4,5]
#Therefore we have 5 experiments this time.

In [ ]:
#The fourth loop, loop over epsilon
#for epsilon_value in EPISLON_LIST:
#On the second cell, we've seen that EPISLON_LIST = [0.01, 0.001]
#That's 2 experiments


In [ ]:
#The Fifth Loop, loop over Entanglement
#for entanglement in ENTANGLEMENT_LIST:
#On the second cell, we've seen that ENTANGLEMENT_LIST:[['linear','full','circular']
#Therefore, we've 3 experiments this time.


#So , that's a total of 4x5x2x3 = 120 models per splits.
#Including the 21 samples, it will be 21x120 = 2520 models.
#With N_REPEATS being 10, it'll be 1260x10 = 25,200 QSVR trainings.

#print(f'C:{C_value} feature_map_reps:{feature_map_reps} entanglement:{entanglement}')
#Used to monitor the process.

'''qsvc = reconfig_quantum_kernel_qsvc(feature_dimension=NUM_FEATURES,
                                      C=C_value,
                                      reps=feature_map_reps,
                                      entangle=entanglement) '''
#We've looked it earlier.Internally it performs 
#                     ZZFeatureMap --> Quantum Kernel --> QSVC
#Every iteration builds a completely new quantum classifier.

#predict_train, predict_test = train_qsvc(qsvc, X_train, y_train, X_test)
#training to predict : Training for Learning and testing for generalization.

#all_preds = y_scaler.inverse_transform(all_preds.reshape(-1, 1))
#all_targets = y_scaler.inverse_transform(all_targets.reshape(-1, 1))
#Previously because of MinMaxScaler, the range was from -1,1. Now it changes back to 0,1
#Everything under #Some Conversions comes under this category.

## save data
''' new_row = {'C': C_value,
                'reps': feature_map_reps,
                'epsilon' : epsilon_value
                'entanglement': entanglement,
                'element test': element_test,
                'actual test': np.array(all_targets).flatten(),
                'predicted test': np.array(all_preds).flatten(),
                'element train': element_train,
                'actual train': np.array(all_targets_train).flatten(),
                'predicted train': np.array(all_preds_train).flatten(),
                #'R2 test': r2_score(y_test, predict_test),
                'R2 train': r2_score(y_train, predict_train),
                            }'''
#This Dictionary stores everything: C, reps, epsilon, entanglement, test material, prediction, everything is preserved.
#Note; only R2 test is not used here, which deepens the idea that the code was originally a regression.
#Because classification papers almost never report R².

#df.loc[len(df)] = new_row: Adds a new row to the end of a dataframe.
#Every experiment adds another row.

#df.to_csv(file_name, index=False)  # update csv every loop
#The result is saved and updated to a csv file everytime a run is completed.

#df.at[0, "info"]
#Adds extra data into the first row.


